# Распознавание штрихкода на картинке

Ниже подгружаем датасет в котором хранятся картинки с видимым штрих кодом и нет

Для начала предполагается выявление штрих кода на картинке с помощью компьютерного зрения

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kniazandrew/ru-goods-barcodes")

print("Path to dataset files:", path)

C:\Users\kseny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\kseny\.cache\kagglehub\datasets\kniazandrew\ru-goods-barcodes\versions\1


In [2]:
import pandas as pd

# Загрузка TSV-файла
df = pd.read_csv('C:\\Users\\kseny\\.cache\\kagglehub\\datasets\\kniazandrew\\ru-goods-barcodes\\versions\\1\\goods-barcodes\\annotation.tsv', sep='\t')

# Удаление строк, где есть хотя бы одно null значение
df = df.dropna()

# Функция для извлечения числовых значений из строки
def split_position(position):
    # Удаляем скобки и разделяем значения по запятой
    x, y = position.strip('()').split(',')
    return int(x), int(y)

# Применяем функцию к столбцу с позициями и создаем два новых столбца
df[['x1', 'y1']] = df['p1'].apply(split_position).apply(pd.Series)

# Удаляем исходный столбец с позициями
df = df.drop(columns=['p1'])

# Применяем функцию к столбцу с позициями и создаем два новых столбца
df[['x2', 'y2']] = df['p2'].apply(split_position).apply(pd.Series)

# Удаляем исходный столбец с позициями
df = df.drop(columns=['p2'])

# Сохраняем результат в новый TSV-файл
df.to_csv('data.tsv', sep='\t', index=False)

In [3]:
import pandas as pd
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Создаем папку для сохранения бинаризированных изображений
output_folder = 'binarized_images'
os.makedirs(output_folder, exist_ok=True)

# Функция для бинаризации изображения
def binarize_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Преобразуем в grayscale
    _, binary = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)  # Бинаризация
    return binary


# Загрузка данных из TSV файла
data = pd.read_csv('data.tsv', sep='\t')

# Предположим, что колонки в TSV файле называются: 'path', 'barcode', 'x1', 'y1', 'x2', 'y2'
# Где (x1, y1) - левый верхний угол, (x2, y2) - правый нижний угол

# Загрузка изображений и подготовка данных
images = []
bboxes = []

for index, row in data.iterrows():
    filename = row['filename']
    img = cv2.imread('images\\' + filename)
    img = cv2.resize(img, (1000, 1000))  # Приводим изображения к одному размеру
    
    # Бинаризация изображения
    binary_img = binarize_image(img)
    images.append(binary_img)
    
    # Сохраняем бинаризированное изображение для проверки
    cv2.imwrite(os.path.join(output_folder, filename), binary_img)
    
    # Нормализуем координаты bounding box
    x1 = row['x1'] / img.shape[1]
    y1 = row['y1'] / img.shape[0]
    x2 = row['x2'] / img.shape[1]
    y2 = row['y2'] / img.shape[0]
    
    bboxes.append([x1, y1, x2, y2])

images = np.array(images, dtype='float32') / 255.0
bboxes = np.array(bboxes, dtype='float32')

# Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(images, bboxes, test_size=0.2, random_state=42)

In [4]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Создание модели
def create_model(input_shape):
    inputs = Input(shape=input_shape)
    
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    outputs = Dense(4, activation='sigmoid')(x)  # 4 выхода: x1, y1, x2, y2
    
    model = Model(inputs, outputs)
    return model

input_shape = (1000, 1000, 1)  # Теперь изображения имеют размер 1000x1000 и 1 канал (grayscale)
model = create_model(input_shape)
model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1000, 1000, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 1000, 1000, 32) │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 500, 500, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 500, 500, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 250, 250, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 250, 250, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 125, 125, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 125, 125, 256)  │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 62, 62, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 984064)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │   503,841,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 504,231,172 (1.88 GB)

 Trainable params: 504,231,172 (1.88 GB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=8)

Epoch 1/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 1622s 30s/step - accuracy: 0.0458 - loss: 0.2189 - val_accuracy: 0.0000e+00 - val_loss: 0.2905
Epoch 2/20
53/54 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - accuracy: 0.0000e+00 - loss: 0.2537

KeyboardInterrupt: 